# Partie C — Le produit gagnant

Source des données : Inside Airbnb (licence CC BY 4.0), snapshot : 2026-09-17.
Ville : Lyon.
Responsable : Membre 3. En appui : Membre 1 (C1, C4 selon Gantt), Membre 2 (C2, C3, C4).


> Ce notebook lit les tables nettoyées du Membre 1 (`data_clean/listings_propre.parquet`, produites par le notebook A, signature MD5 reproduite). Sans ce fichier, il se rabat sur les données brutes avec un nettoyage minimal, uniquement pour une exploration provisoire.

In [ ]:
import sys
sys.path.append("..")
from src.config import *
import re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.equipements import MOTIFS, parse_amenities, ajouter_indicatrices
from src.primes import (perimetre_logements_entiers, ajouter_classe_capacite, classes_valides,
                        tableau_primes)
np.random.seed(SEED)
print("Ville :", VILLE)
print("Dossier données :", DOSSIER, "| existe :", DOSSIER.exists())

## C0 — Chargement et périmètre

**Périmètre retenu : logements entiers, actifs, avec un prix publié.**
*Justification :* mélanger chambres et logements entiers fausse toutes les comparaisons (la taille et le standing diffèrent). Un logement entier est aussi ce que vise un investisseur. Un prix publié est nécessaire pour calculer les primes de prix, et une annonce active garantit que le revenu estimé est observable.
*Définition de « active » (celle de A1) :* au moins 1 avis sur les 12 derniers mois **et** des jours disponibles dans l'année. Sur ce périmètre, elle donne presque exactement le même résultat que « occupation estimée > 0 » (3 242 contre 3 244 annonces), car avoir un prix publié suppose déjà que l'annonce est réservable.

In [ ]:
COLS = ["id", "room_type", "price", "accommodates", "bedrooms", "amenities",
        "neighbourhood_cleansed", "estimated_occupancy_l365d", "estimated_revenue_l365d",
        "review_scores_rating", "host_is_superhost", "minimum_nights",
        "calculated_host_listings_count", "number_of_reviews_ltm", "availability_365"]

FICHIER_PROPRE = CLEAN / "listings_propre.parquet"   # nom convenu avec l'équipe pour A6


def prix_en_nombre(v):
    """Conversion provisoire de price. Formats vus dans les données brutes de Lyon :
    '$1,250.00', '142,50 $' (virgule décimale), '136 dollars', 'CAD 184', '118.00', '-$90.00'.
    Le signe est ignoré (erreur de signe probable) : A5 tranchera pour la table nettoyée."""
    if pd.isna(v):
        return np.nan
    s = re.sub(r"[^\d.,]", "", str(v))
    if not s:
        return np.nan
    if "." not in s and re.search(r",\d{1,2}$", s):   # virgule décimale
        s = s.replace(",", ".")
    else:                                              # virgule = séparateur de milliers
        s = s.replace(",", "")
    return float(s)


if FICHIER_PROPRE.exists():
    listings = pd.read_parquet(FICHIER_PROPRE)
    SOURCE = "table nettoyée (A6)"
else:
    listings = pd.read_csv(DOSSIER / "listings.csv.gz", usecols=COLS)
    # Nettoyage minimal et provisoire : A3 et A5 feront le vrai traitement
    listings = listings.drop_duplicates(subset="id", keep="first")
    listings["price"] = listings["price"].map(prix_en_nombre)
    listings["neighbourhood_cleansed"] = listings["neighbourhood_cleansed"].str.strip().str.lower()
    listings = listings[listings["accommodates"].between(1, 16)]
    SOURCE = "données brutes (exploration provisoire)"

if not pd.api.types.is_bool_dtype(listings["host_is_superhost"]):
    # écritures vues : t/f, true/false, 1/0, yes/no, vrai/faux ; A3 fera le traitement complet
    listings["host_is_superhost"] = (
        listings["host_is_superhost"].astype(str).str.strip().str.lower()
        .map({"t": True, "true": True, "1": True, "yes": True, "vrai": True,
              "f": False, "false": False, "0": False, "no": False, "faux": False}))

print(SOURCE, "|", listings.shape)

# Entonnoir du périmètre
# Définition de A1 : au moins 1 avis sur 12 mois et des jours disponibles
listings["annonce_active"] = (listings["number_of_reviews_ltm"] >= 1) & (listings["availability_365"] > 0)
entiers = listings["room_type"].astype(str).str.lower().str.contains("entire")
avec_prix = listings["price"].notna() & (listings["price"] > 0)
print("Annonces au total        :", len(listings))
print("... logements entiers    :", int(entiers.sum()))
print("... avec un prix publié  :", int((entiers & avec_prix).sum()))
perimetre = perimetre_logements_entiers(listings, "annonce_active")
print("... actives (périmètre)  :", len(perimetre))

## C1 — La prime des équipements

### C1.1 — Choix des équipements
On part de 12 candidats plausibles pour un investisseur (voir `src/equipements.py`). On retient ceux qui sont **ni trop rares, ni quasi universels** (présence entre 5 % et 95 %), car un équipement que presque tout le monde a (ou n'a pas) ne permet aucune comparaison, et qui ont assez d'annonces pour une comparaison à capacité égale (au moins 2 classes de capacité avec 10 annonces équipées et 10 non équipées).

In [ ]:
perimetre = ajouter_indicatrices(perimetre)
perimetre = ajouter_classe_capacite(perimetre)

selection = pd.DataFrame({
    "part_equipees_%": (perimetre[list(MOTIFS)].mean() * 100).round(1),
    "classes_comparables": [classes_valides(perimetre, eq) for eq in MOTIFS],
})
selection["retenu"] = (selection["part_equipees_%"].between(5, 95)
                       & (selection["classes_comparables"] >= 2))
display(selection.sort_values("part_equipees_%", ascending=False))

EQUIPEMENTS_RETENUS = selection.index[selection["retenu"]].tolist()
print(len(EQUIPEMENTS_RETENUS), "équipements retenus :", EQUIPEMENTS_RETENUS)
if not 8 <= len(EQUIPEMENTS_RETENUS) <= 12:
    print("Attention : l'énoncé demande 8 à 12 équipements. Ajuster les seuils ou les motifs.")

*Liste finale validée en équipe (à compléter) :*

### C1.2 à C1.5 — Naïve, annonces comparables, régression
Trois méthodes sur les mêmes équipements :
1. **naïve** : écart des médianes équipées / non équipées ;
2. **annonces comparables** : même comparaison à l'intérieur de chaque classe de capacité (1-2, 3-4, 5-6, 7+), moyenne pondérée par la taille des groupes (annexe E) ;
3. **régression** : log(prix) sur tous les équipements, la capacité, les chambres et le quartier ; prime = exp(coefficient) − 1.

In [ ]:
tab = tableau_primes(perimetre, EQUIPEMENTS_RETENUS)
tab = tab.sort_values("comparable_revenu", ascending=False)
display((tab * 100).round(1).rename(columns=lambda c: c + " (%)"))

In [ ]:
# Les trois primes de prix côte à côte : voir l'écart entre méthodes (C1.6)
graph = tab[["naive_prix", "comparable_prix", "regression_prix"]].mul(100)
graph.columns = ["Naïve", "À capacité égale", "Régression"]
ax = graph.plot.barh(figsize=(9, 6))
ax.axvline(0, color="grey", linewidth=0.8)
ax.set_xlabel("Prime de prix (%)")
ax.set_ylabel("Équipement")
ax.set_title("La prime naïve surestime l'effet des équipements")
plt.tight_layout()
plt.show()

### C1 (suite) — Robustesse des primes
On vérifie que les conclusions ne dépendent pas d'un choix : (1) la définition de « annonce active », (2) la présence des prix extrêmes signalés en A5 (`prix_extreme`). Un équipement dont la prime change de signe ou d'ordre de grandeur ne doit pas fonder une recommandation.

In [ ]:
cles = ["a_climatisation", "a_lave_vaisselle", "a_vue", "a_foyer", "a_parking", "a_arrivee_autonome"]
cles = [e for e in cles if e in EQUIPEMENTS_RETENUS]

variantes = {"référence (A1)": perimetre}
alt = ajouter_classe_capacite(ajouter_indicatrices(
    perimetre_logements_entiers(listings.assign(alt=listings["estimated_occupancy_l365d"] > 0), "alt")))
variantes["active = occupation estimée > 0"] = alt
if "prix_extreme" in perimetre:
    variantes["sans prix extrêmes"] = perimetre[~perimetre["prix_extreme"]]

lignes = {}
for nom, base in variantes.items():
    t = tableau_primes(base, cles)
    lignes[(nom, "comparable_prix")] = t["comparable_prix"]
    lignes[(nom, "regression_prix")] = t["regression_prix"]
display((pd.DataFrame(lignes).T * 100).round(1))

### C1.6 — Lecture
**Pourquoi la prime naïve surestime.** Les grands logements bien situés cumulent les équipements *et* les prix élevés : comparer simplement les annonces équipées et non équipées mesure en partie la taille et le standing. Exemple du lave-vaisselle : +43 % (naïve), +22 % (à capacité égale), +12 % (régression).

**Équipements recommandés** (prime de prix positive avec les trois méthodes) :
- **Climatisation** : +18 % à capacité égale, +14 % en régression ; 25 % des annonces l'ont. Confiance élevée (été chaud à Lyon).
- **Lave-vaisselle** : +22 % et +12 % ; 45 % des annonces. Confiance élevée, équipement de base à prévoir.
- **Vue** : +11 % et +5 % ; 12 % des annonces. Confiance moyenne : la qualité réelle de la vue n'est pas observée.
- **Ascenseur** : +5 % et +4,5 % ; effet modeste, confiance moyenne à faible.
- **Foyer** : +37 % et +23 % (+18 % sans les prix extrêmes), mais seulement 5 % des annonces. Confiance faible : échantillon réduit et sensible aux valeurs extrêmes. Pas de recommandation d'achat.

**Équipements à ne pas recommander** : parking (≈ 0 %), balcon ou terrasse (+3 % puis −1 %), espace de travail (+6 % puis +0,4 %), lave-linge (+7 % puis −0,2 %), machine à café (+8 % puis −0,2 %), arrivée autonome (−1 %, +3 %, +4 % : signe instable). Leur effet est faible ou change de signe d'une méthode à l'autre.

**Prudence.** Les primes de *revenu* (machine à café +137 %, arrivée autonome +108 %) viennent de l'occupation estimée, calculée depuis les avis, pas du prix : elles ne mesurent pas l'effet de l'équipement. Décoration, photos et réactivité de l'hôte ne sont pas observées : ces primes sont des **associations robustes, pas des effets garantis**.

## C2 — L'arbitrage prix / occupation

Déciles du **prix par personne** (et non du prix brut, sinon les déciles les plus chers regroupent les grands biens : on mesurerait la taille, pas le niveau de prix).

In [ ]:
c2 = perimetre[perimetre["accommodates"] > 0].copy()
c2["prix_par_personne"] = c2["price"] / c2["accommodates"]
c2["decile_prix"] = pd.qcut(c2["prix_par_personne"], 10, labels=False, duplicates="drop") + 1

deciles = c2.groupby("decile_prix").agg(
    annonces=("id", "count"),
    prix_pers_median=("prix_par_personne", "median"),
    occupation_mediane=("estimated_occupancy_l365d", "median"),
    revenu_median=("estimated_revenue_l365d", "median"),
).round(1)
display(deciles)

meilleur = deciles["revenu_median"].idxmax()
print(f"Décile de prix au revenu médian le plus élevé : {meilleur} "
      f"(prix médian par personne : {deciles.loc[meilleur, 'prix_pers_median']} par nuit)")

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(deciles.index, deciles["occupation_mediane"], marker="o", color="tab:blue")
ax1.set_xlabel("Décile de prix par personne (1 = le moins cher)")
ax1.set_ylabel("Occupation médiane estimée (nuits sur 12 mois)", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(deciles.index, deciles["revenu_median"], marker="s", color="tab:orange")
ax2.set_ylabel("Revenu médian estimé (par an, devise locale)", color="tab:orange")
ax1.set_title("L'occupation baisse quand le prix monte : où le revenu est-il maximal ?")
plt.tight_layout()
plt.show()

# Fiabilité : plafond de l'occupation estimée
plafond = perimetre["estimated_occupancy_l365d"].max()
part_plafond = (perimetre["estimated_occupancy_l365d"] == plafond).mean()
print(f"Occupation maximale observée : {plafond} nuits ; annonces au plafond : {part_plafond:.1%}")

### C2 (suite) — Déciles de prix à l'intérieur de chaque classe de capacité
Le prix par personne reste lié à la taille du bien (un studio pour 2 coûte plus cher par personne qu'un 6 personnes). Pour comparer des biens comparables, on refait les déciles **dans chaque classe de capacité** : le décile 10 devient « parmi les plus chers de sa catégorie ».

In [ ]:
c2["decile_dans_classe"] = (c2.groupby("classe_capacite", observed=True)["prix_par_personne"]
                            .transform(lambda s: pd.qcut(s, 10, labels=False, duplicates="drop") + 1))
par_decile = c2.groupby("decile_dans_classe").agg(
    annonces=("id", "count"),
    occupation_mediane=("estimated_occupancy_l365d", "median"),
    revenu_median=("estimated_revenue_l365d", "median")).round(1)
display(par_decile)
print("Décile (relatif à sa classe) au revenu médian le plus élevé :", par_decile["revenu_median"].idxmax())

In [ ]:
# Zone de prix pour la classe de capacité visée (déciles 6 à 8 de la classe)
classe_visee = c2[c2["classe_capacite"] == "3-4"]
zone_prix = classe_visee[classe_visee["decile_dans_classe"].isin([6, 7, 8])]["price"]
print(f"Classe 3-4 personnes : prix médian {classe_visee['price'].median():.0f} € par nuit")
print(f"Déciles 6 à 8 de la classe : de {zone_prix.quantile(0.25):.0f} à {zone_prix.quantile(0.75):.0f} € par nuit "
      f"(médiane {zone_prix.median():.0f} €)")

*Lecture :*
- **Sur tout le périmètre**, le revenu médian augmente avec le prix par personne (7 952 € au décile 1, 13 461 € au décile 10) alors que l'occupation médiane baisse (108 puis 66 nuits) : la hausse de prix compense la baisse d'occupation. Mais le prix par personne mélange petits et grands biens.
- **À l'intérieur de chaque classe de capacité**, le revenu médian culmine au **8e décile** (14 943 €, 96 nuits) puis recule (12 900 € au 9e décile, 13 628 € au 10e avec seulement 54 nuits). Au-delà d'un prix situé dans le haut de la fourchette de sa catégorie, l'occupation chute plus vite que le prix ne monte. **Zone conseillée : déciles 6 à 8 de sa classe**, soit environ 138 à 166 € la nuit pour 3 à 4 personnes.
- **Fiabilité.** L'occupation estimée est plafonnée à 255 nuits et 15,3 % des annonces du périmètre y sont : leur occupation et leur revenu sont sous-estimés, et on ne distingue plus les annonces les plus demandées. Comme l'occupation vient des avis, une annonce chère qui reçoit moins d'avis paraît moins occupée. La zone de prix est donc une indication de prudence, pas un optimum mesuré.
- **Comment l'occupation est fabriquée (vérifié sur nos données) :** pour 89 % des annonces sous le plafond, occupation estimée = avis sur 12 mois / 0,5 x 3 nuits ; pour les autres, 3 nuits est remplacé par la durée minimale de séjour. L'occupation et le revenu sont proportionnels au nombre d'avis récents.

## C3 — Le profil des annonces les plus rentables

In [ ]:
c3 = perimetre.copy()
if "nb_equipements" not in c3:   # déjà calculé dans la table nettoyée (A3)
    c3["nb_equipements"] = c3["amenities"].map(lambda v: len(parse_amenities(v)))
seuil = c3["estimated_revenue_l365d"].quantile(0.90)
c3["groupe"] = np.where(c3["estimated_revenue_l365d"] >= seuil, "Top 10 %", "Reste")

numeriques = ["accommodates", "bedrooms", "price", "nb_equipements", "review_scores_rating",
              "minimum_nights", "calculated_host_listings_count", "estimated_occupancy_l365d"]
comparaison = c3.groupby("groupe")[numeriques].median().T
comparaison.loc["host_is_superhost (%)"] = (
    c3.groupby("groupe")["host_is_superhost"].mean().mul(100).round(1))
display(comparaison.round(2))

top_quartiers = (c3[c3["groupe"] == "Top 10 %"]["neighbourhood_cleansed"]
                 .value_counts(normalize=True).head(5).mul(100).round(1))
print("Quartiers les plus représentés dans le top 10 % (%) :")
display(top_quartiers)

**Les trois ingrédients du succès** (top 10 % comparé au reste) :
1. **Un logement plus grand et plus cher** : capacité de 4 personnes et 2 chambres (contre 2 et 1), prix médian de 204 € la nuit (contre 114 €).
2. **Une forte réservation, soutenue par la flexibilité et la fiabilité** : occupation médiane de 252 nuits (contre 78), durée minimale d'une nuit (contre 2) et statut Superhost pour 57,5 % des annonces (contre 37,9 %).
3. **Une offre mieux équipée et plus professionnelle** : 40 équipements (contre 32) et hôtes gérant 3 annonces (contre 2).
La note moyenne (4,84 contre 4,81) ne distingue pas les deux groupes : presque toutes les annonces sont bien notées.

**À nuancer :** l'occupation médiane du top 10 % (252 nuits) est presque au plafond de 255 nuits. Le revenu élevé vient donc d'un prix supérieur combiné à une occupation estimée saturée ; ces ingrédients décrivent des annonces très réservées, pas une garantie pour un nouvel entrant.

**Biais du survivant.** Le fichier ne contient que les annonces encore présentes ; celles qui ont échoué ont disparu. Les revenus observés, surtout ceux du top 10 %, sont donc ceux de survivants et surestiment ce qu'un nouvel investisseur obtiendra. Pour la note de décision, on s'appuie sur la médiane et sur le scénario bas plutôt que sur le haut du classement.

### C3 (suite) — Définition opérationnelle du produit gagnant
**Produit gagnant = la combinaison (type de bien, capacité, quartier, zone de prix, équipements) qui maximise le revenu net attendu, avec un échantillon assez grand pour être fiable et une faisabilité réglementaire.**
Règle de décision : on classe les combinaisons capacité × quartier ayant au moins 30 annonces actives, par revenu médian ; on regarde l'occupation (et la part d'annonces au plafond) pour ne pas confondre prix élevé et vraie demande.
*Les chiffres sont provisoires tant que la table nettoyée (A6) n'existe pas.*

In [ ]:
MIN_ANNONCES = 30
cellules = (perimetre.groupby(["classe_capacite", "neighbourhood_cleansed"], observed=True)
            .agg(annonces=("id", "count"),
                 prix_median=("price", "median"),
                 occupation_mediane=("estimated_occupancy_l365d", "median"),
                 revenu_median=("estimated_revenue_l365d", "median"),
                 part_au_plafond=("estimated_occupancy_l365d", lambda s: (s == s.max()).mean()))
            .query("annonces >= @MIN_ANNONCES")
            .sort_values("revenu_median", ascending=False).round(2))
display(cellules.head(10))

In [ ]:
# Profil retenu : à valider par l'équipe (voir « Produit retenu » ci-dessous)
PROFIL = {"classe_capacite": "3-4",
          "arrondissements": ["1er Arrondissement", "2e Arrondissement", "5e Arrondissement"]}   # les trois premiers du classement B3 Q3
profil = perimetre[(perimetre["classe_capacite"] == PROFIL["classe_capacite"])
                   & perimetre["neighbourhood_cleansed"].isin(PROFIL["arrondissements"])]
print(f"Profil retenu : {len(profil)} annonces ({PROFIL['classe_capacite']} personnes, "
      f"{', '.join(a.split()[0] for a in PROFIL['arrondissements'])})")
print(f"Prix médian {profil['price'].median():.0f} € la nuit | occupation médiane "
      f"{profil['estimated_occupancy_l365d'].median():.0f} nuits | revenu médian estimé "
      f"{profil['estimated_revenue_l365d'].median():,.0f} € par an".replace(",", " "))
equip = profil[EQUIPEMENTS_RETENUS].mean().mul(100).round(0).sort_values(ascending=False)
print("Part des annonces du profil équipées (%) :", equip.to_dict())

**Produit retenu (provisoire, à valider par l'équipe) :** logement entier de **3 à 4 personnes (1 à 2 chambres)** dans les **1er, 2e et 5e arrondissements**, avec un **prix de lancement de 140 à 165 € la nuit** (déciles 6 à 8 de la classe). **Équipements prioritaires :** climatisation et lave-vaisselle (confiance élevée), vue si elle existe (confiance moyenne).

*Pourquoi cette combinaison :*
- les 1er, 2e et 5e arrondissements sont les trois premiers du classement de B3 Q3 (revenu, occupation, revenu prudent, concurrence) et ont chacun plus de 140 annonces de 3 à 4 personnes, donc des médianes fiables ;
- les 3 à 4 personnes donnent un revenu médian de 11 808 € par an, supérieur aux 1-2 personnes (9 890 €) ; les 5 à 6 personnes montent à 14 010 € mais ne comptent que 389 annonces et coûteraient plus cher à acheter (coût non mesuré dans les données) ;
- le 5e arrondissement offre le meilleur compromis : revenu médian de 13 049 € pour 8,5 annonces par 1 000 habitants, contre 21 dans le 1er arrondissement, plus rémunérateur mais bien plus concurrentiel.

## C4 — Du revenu à la rentabilité

Les données ne contiennent aucun coût. Les hypothèses ci-dessous sont documentées dans `docs/hypotheses_couts.md` (sources et date de consultation). `[À SOURCER]` signale une hypothèse non encore sourcée.
- **Commission Airbnb :** depuis le 13/10/2026, frais unique de 15,5 % à la charge de l'hôte en France et dans l'Espace économique européen (ancien modèle : 3 % hôte et 14 à 16 % voyageur). Là où la TVA s'applique, le taux affiché l'inclut déjà.
- **Séjours :** Inside Airbnb estime les réservations par `avis sur 12 mois / 0,5` ; on en déduit le nombre de ménages.
- **Ménage :** le revenu estimé ne contient pas les frais de ménage facturés au voyageur, qui compensent en partie ce coût : le net calculé ici est donc prudent.
- **Non modélisé :** achat et crédit, taxe foncière, charges de copropriété, vacance, prélèvements sociaux, temps de gestion.

In [ ]:
HYP = {
    "frais_plateforme_pct": 0.155,      # frais unique côté hôte (France, depuis le 13/10/2026), TVA incluse si applicable
    "tva_supplementaire": 0.0,          # sensibilité : 0.20 si la TVA s'ajoutait aux frais (sources discordantes hors France)
    "taux_avis": 0.5,                   # Inside Airbnb : taux d'avis de 50 % (insideairbnb.com/data-assumptions)
    "menage_par_sejour": 55.0,          # T2/T3 : 40 à 90 € (Houst) ; Lyon : 35 à 70 € par check-out (GuestReady)
    "consommables_par_nuit": 3.0,       # [À SOURCER]
    "charges_fixes_annuelles": 1800.0,  # [À SOURCER] énergie, internet, entretien
    "assurance_annuelle": 375.0,        # MRH bailleur meublé ou saisonnier : 250 à 500 € par an (France-Epargne d'après Selectra, 2026)
}
# Fiscalité : micro-BIC, meublé de tourisme NON classé (Service-Public) ; la tranche (tmi) est une hypothèse
FISCALITE = {"abattement": 0.30, "plafond_recettes": 15000, "tmi": 0.30}
# Coût d'opportunité : louer le même bien en longue durée
LONGUE_DUREE = {"surface_m2": 45,            # [À SOURCER] T2/T3 pour 3 à 4 personnes (surface moyenne OLL : 55 à 61 m²)
                "loyer_m2_oll_zone1": 14.10,  # OLL (Urbalyon) 2023, Lyon zone 1 (centre), parc privé, hors charges
                "loyer_m2_meuble": 17.0}      # source privée : ordre de grandeur du meublé

loyer_oll = LONGUE_DUREE["surface_m2"] * LONGUE_DUREE["loyer_m2_oll_zone1"] * 12
loyer_meuble = LONGUE_DUREE["surface_m2"] * LONGUE_DUREE["loyer_m2_meuble"] * 12


def scenario(q, hyp=HYP):
    revenu = profil["estimated_revenue_l365d"].quantile(q)
    nuits = profil["estimated_occupancy_l365d"].quantile(q)
    sejours = profil["number_of_reviews_ltm"].quantile(q) / hyp["taux_avis"]
    frais = revenu * hyp["frais_plateforme_pct"] * (1 + hyp["tva_supplementaire"])
    menage = sejours * hyp["menage_par_sejour"]
    conso = nuits * hyp["consommables_par_nuit"]
    fixes = hyp["charges_fixes_annuelles"] + hyp["assurance_annuelle"]
    net = revenu - frais - menage - conso - fixes
    impot = revenu * (1 - FISCALITE["abattement"]) * FISCALITE["tmi"] if revenu <= FISCALITE["plafond_recettes"] else np.nan
    return {"revenu_brut": round(revenu), "nuits": round(nuits), "sejours": round(sejours),
            "frais_plateforme": round(frais), "menage": round(menage),
            "net_avant_impot": round(net), "impot_micro_bic": impot}


lignes = []
for nom, q in [("bas", 0.25), ("médian", 0.50), ("haut", 0.75)]:
    ligne = {"scénario": nom, **scenario(q)}
    ligne["loyer_longue_duree_OLL"] = round(loyer_oll)
    ligne["loyer_longue_duree_meuble"] = round(loyer_meuble)
    lignes.append(ligne)
resultat_c4 = pd.DataFrame(lignes).set_index("scénario")
display(resultat_c4.round(0))

In [ ]:
# Sensibilité du scénario médian aux hypothèses les plus incertaines
variantes = {
    "hypothèses de base": {},
    "commission + 20 % de TVA (18,6 %)": {"tva_supplementaire": 0.20},
    "ménage à 70 € par séjour": {"menage_par_sejour": 70.0},
    "charges fixes + 50 %": {"charges_fixes_annuelles": HYP["charges_fixes_annuelles"] * 1.5},
    "assurance basse (PNO, 128 €)": {"assurance_annuelle": 128.0},
}
sensibilite_c4 = pd.Series({nom: scenario(0.50, {**HYP, **chg})["net_avant_impot"] for nom, chg in variantes.items()},
                           name="net_avant_impot_median")
display(sensibilite_c4.to_frame())

med, bas, haut = resultat_c4.loc["médian"], resultat_c4.loc["bas"], resultat_c4.loc["haut"]
fr = lambda x: f"{x:,.0f}".replace(",", " ")
print(f"Profil retenu ({len(profil)} annonces) : revenu net médian avant impôt de {fr(med['net_avant_impot'])} € par an "
      f"(fourchette {fr(bas['net_avant_impot'])} à {fr(haut['net_avant_impot'])} €), pour un revenu brut médian de {fr(med['revenu_brut'])} €.")
print(f"Louer le même bien en longue durée rapporterait environ {fr(loyer_oll)} € par an (OLL, hors meublé) "
      f"à {fr(loyer_meuble)} € (meublé estimé), sans commission ni ménage.")
if med["net_avant_impot"] > loyer_meuble:
    print("Le net médian de la courte durée dépasse le loyer meublé : l'effort supplémentaire est rémunéré.")
elif med["net_avant_impot"] > loyer_oll:
    print("Le net médian dépasse le loyer non meublé mais pas le loyer meublé estimé : l'avantage est mince au regard de l'effort, de la réglementation et du risque.")
else:
    print("Le net médian reste inférieur au loyer longue durée : la courte durée ne se justifie pas sur ce seul critère.")

*Lecture :* le scénario médian est la référence ; le scénario bas représente un nouvel entrant sans historique d'avis (biais du survivant, voir C3). Les recettes du scénario médian (14 928 €) frôlent le plafond du micro-BIC (15 000 €) et celles du scénario haut le dépassent : pour ce dernier, l'impôt n'est pas modélisé ici (régime réel).
*Limites :* hypothèses de charges et de consommables non sourcées, prix d'achat inconnu (donc aucun rendement ni taux de rentabilité), commission d'Airbnb issue de la presse, loyer meublé issu d'une source privée. La conclusion se lit donc comme un ordre de grandeur, à reprendre dans la note de décision avec ses fourchettes.

## C5 — Export des tables pour l'application (Membre 2, E1)
L'application lit uniquement des tables nettoyées. On exporte quatre tables dans `data_clean/` (Parquet et CSV) :
- `c_perimetre` : une ligne par annonce du périmètre, **mêmes noms de colonnes que `listings_propre`** (anglais) pour réutiliser `src/primes.py` après filtrage dans l'application ;
- `c_primes`, `c_deciles`, `c_profils` : tables de synthèse en colonnes françaises, prêtes à afficher.
Les libellés français de toutes les colonnes sont dans `src/libelles.py`.

In [ ]:
from src.signature import signature

# 1. Périmètre au niveau annonce (sans identifiant d'hôte ni texte : aucune donnée personnelle)
c_perim = c2.copy()
if "nb_equipements" not in c_perim:
    c_perim["nb_equipements"] = c_perim["amenities"].map(lambda v: len(parse_amenities(v)))
c_perim["hote_multi"] = c_perim["calculated_host_listings_count"] > 1          # provisoire, B1 fait foi
c_perim["au_plafond"] = c_perim["estimated_occupancy_l365d"] == c_perim["estimated_occupancy_l365d"].max()
colonnes = (["id", "neighbourhood_cleansed", "room_type", "accommodates", "bedrooms", "classe_capacite",
             "price", "prix_par_personne", "decile_prix", "decile_dans_classe",
             "estimated_occupancy_l365d", "estimated_revenue_l365d", "review_scores_rating",
             "host_is_superhost", "minimum_nights", "calculated_host_listings_count", "hote_multi",
             "nb_equipements", "prix_extreme", "au_plafond"] + EQUIPEMENTS_RETENUS)
c_perimetre = c_perim[[c for c in colonnes if c in c_perim]].copy()
c_perimetre["classe_capacite"] = c_perimetre["classe_capacite"].astype(str)
c_perimetre = c_perimetre.sort_values("id").reset_index(drop=True)

# 2. Primes d'équipements (tableau de C1)
c_primes = tab.reset_index().rename(columns={
    "naive_prix": "prime_naive_prix", "naive_revenu": "prime_naive_revenu",
    "comparable_prix": "prime_comparable_prix", "comparable_revenu": "prime_comparable_revenu",
    "regression_prix": "prime_regression_prix"})
c_primes["signes_concordants"] = (np.sign(c_primes["prime_comparable_prix"])
                                  == np.sign(c_primes["prime_regression_prix"]))
c_primes = c_primes.sort_values("equipement").reset_index(drop=True)

# 3. Déciles de prix (C2) : deux bases empilées
d1 = deciles.reset_index().rename(columns={"decile_prix": "decile", "prix_pers_median": "prix_pers_median"})
d1.insert(0, "base_decile", "prix par personne (tout le périmètre)")
d2 = par_decile.reset_index().rename(columns={"decile_dans_classe": "decile"})
d2.insert(0, "base_decile", "prix par personne dans la classe de capacité")
c_deciles = pd.concat([d1, d2], ignore_index=True)

# 4. Profils capacité x arrondissement (tableau « produit gagnant »)
c_profils = (cellules.reset_index()
             .rename(columns={"neighbourhood_cleansed": "arrondissement"}))
c_profils["classe_capacite"] = c_profils["classe_capacite"].astype(str)
c_profils = c_profils.sort_values(["classe_capacite", "arrondissement"]).reset_index(drop=True)

tables = {"c_perimetre": c_perimetre, "c_primes": c_primes, "c_deciles": c_deciles, "c_profils": c_profils}
for nom, t in tables.items():
    t.to_parquet(CLEAN / f"{nom}.parquet", index=False)
    t.to_csv(CLEAN / f"{nom}.csv", index=False, encoding="utf-8-sig")
    signature(t, nom)

*Vérification :* la signature de chaque table est affichée ci-dessus ; le Membre 2 doit retrouver les mêmes empreintes après avoir relancé A puis C avec pandas 2.3.3.